In [1]:
import os
import csv
import calendar

import re
import pandas as pd
from Bio import SeqIO

First we will parse genbank file, filter sequences with ambiguous characters

In [10]:
recs = list(SeqIO.parse("raw/Bastrovirus_22102026.gb", "genbank"))

#list with ambiguous nucleotides
ambig_nt = ['n',  # A, T, G, C 
            'r', # Purine (A or G)
            'y', # Pyrimidine (T or C)
            'k', # Keto (G or T)
            'm', # Amino (A or C)
            's', # Strong interaction (3 H bonds) (G or C)
            'w', # Weak interaction (2 H bonds) (A or T)
            'b', # Not A (C or G or T)
            'd', # Not C (A or G or T)
            'h', # Not G (A or C or T)
            'v' # Not T or U (A or C or G)
            ]


recs_noamb = []
total_amb = 0

    for nt in ambig_nt:
        total_amb += rec.seq.lower().count(nt)
    if total_amb/len(rec.seq)< 0.08:
        recs_noamb.append(rec)
    else:
        print(f"{rec.name} has {total_amb} ({round(100*total_amb/len(rec.seq),2)}%) ambiguous nucleotides")
print(f"There are {len(recs_noamb)} sequences with <1% ambiguous nucleotides")

print(f"There are {len(not_in_meta)} absent in Astroviridae metadata")


SeqIO.write(recs_noamb, "raw/Bastrovirus_22102026_lessamb.gb", "genbank")
SeqIO.write(recs_noamb, "raw/Bastrovirus_22102026_lessamb.fna", "fasta")

C:\Users\alxva\AppData\Local\Temp\ipykernel_21476\2143751514.py:2: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  meta = pd.read_csv(meta_path,sep="\t")


OQ802730 has 547 (9.1%) ambiguous nucleotides
OQ802729 has 547 (9.31%) ambiguous nucleotides
OQ802713 has 547 (9.9%) ambiguous nucleotides
OR439362 has 547 (9.79%) ambiguous nucleotides
OR552431 has 566 (9.98%) ambiguous nucleotides
OR552430 has 570 (9.77%) ambiguous nucleotides
OQ599876 has 570 (8.15%) ambiguous nucleotides
ON304005 has 570 (9.68%) ambiguous nucleotides
OM953852 has 743 (12.94%) ambiguous nucleotides
MT568535 has 743 (14.17%) ambiguous nucleotides
MT766313 has 743 (14.11%) ambiguous nucleotides
LC577872 has 743 (12.44%) ambiguous nucleotides
LC577871 has 743 (13.42%) ambiguous nucleotides
LC577870 has 743 (14.26%) ambiguous nucleotides
MG693175 has 743 (13.08%) ambiguous nucleotides
NC_035758 has 743 (11.72%) ambiguous nucleotides
KU318321 has 743 (11.72%) ambiguous nucleotides
KU318320 has 744 (11.81%) ambiguous nucleotides
KU318319 has 746 (12.35%) ambiguous nucleotides
KU318318 has 746 (12.26%) ambiguous nucleotides
KU318317 has 746 (11.77%) ambiguous nucleotides
K

28

In [19]:
def standardize_date(date_str):
    """
    Convert various date formats to ISO 8601-like string:
      - YYYY-MM-DD if day available
      - YYYY-MM if only month available
      - YYYY if only year available
    Handles ranges (e.g., 08-Jul-2014/11-Oct-2016) by joining with '_'.
    """
    if pd.isna(date_str):
        return 'NA'
    s = str(date_str).strip()

    # Handle date range: split on '/' and standardize each part, then join with '_'
    if '/' in s and not re.match(r'^\d{4}$', s) and not re.match(r'^\d{4}-\d{2}$', s):
        parts = s.split('/')
        std_parts = [standardize_date(p) for p in parts]
        return '_'.join(std_parts)

    # DD-MMM-YYYY (e.g., 13-Jun-2024)
    match = re.match(r'^(\d{1,2})-([A-Za-z]{3})-(\d{4})$', s)
    if match:
        day, mon, year = match.groups()
        month_num = list(calendar.month_abbr).index(mon.capitalize())
        return f"{year}-{month_num:02d}-{int(day):02d}"

    # DD.MM.YYYY (e.g., 02.10.2019)
    match = re.match(r'^(\d{1,2})\.(\d{1,2})\.(\d{4})$', s)
    if match:
        day, mon, year = match.groups()
        return f"{year}-{int(mon):02d}-{int(day):02d}"

    # YYYY-MM (e.g., 2019-10)
    match = re.match(r'^(\d{4})-(\d{1,2})$', s)
    if match:
        year, mon = match.groups()
        return f"{year}-{int(mon):02d}"

    # MM-YYYY or MM.YYYY (e.g., 11-2025 or 11.2025)
    match = re.match(r'^(\d{1,2})[-.](\d{4})$', s)
    if match:
        mon, year = match.groups()
        return f"{year}-{int(mon):02d}"

    # MMM-YYYY (e.g., Nov-2025)
    match = re.match(r'^([A-Za-z]{3})-(\d{4})$', s)
    if match:
        mon, year = match.groups()
        month_num = list(calendar.month_abbr).index(mon.capitalize())
        return f"{year}-{month_num:02d}"

    # Just YYYY
    match = re.match(r'^(\d{4})$', s)
    if match:
        return s

    # If nothing matches, clean and return as is
    return clean_field(s)


In [21]:
source_qualifiers = ["strain", "isolate", "geo_loc_name", "country", "collection_date", "isolation_source", "host"]
source_qualifiers_columns = ["Strain", "Isolate", "Geo location", "Country", "Collection date", "Tissue/Specimen/Source", "Host"]




meta_path = "D:/Virology/RNA_viruses/Astrovirus/Astroviridae_database/data/results/30-06-2026/final/Astroviridae_30062026_annotated_MAstV_species.tsv"

meta = pd.read_csv(meta_path,sep="\t")
meta = meta.set_index("Accession")
not_in_meta = []
entries_data = {}
for entry in recs:
    rec_acc = entry.id.split(".")[0]
    if rec_acc not in meta.index:
        not_in_meta.append(rec_acc)
        
        entry_name = entry.id.split(".")[0]
        entries_data[entry_name] = {}
        entries_data[entry_name]['LOCUS'] = entry.name
        entries_data[entry_name]['Version'] = entry.id
        entries_data[entry_name]['GenBank title'] = entry.description
    
        entries_data[entry_name]['Release date'] = entry.annotations['date']
        entries_data[entry_name]['Organism name'] = entry.annotations['organism']
        
        entries_data[entry_name]['Virus Lineage'] = ';'.join(entry.annotations['taxonomy'])
        num_taxa = len(entry.annotations['taxonomy'])
        if num_taxa == 9:
            entries_data[entry_name]['Species'] = entry.annotations['taxonomy'][-1]
        else:
             entries_data[entry_name]['Species'] = 'NA'
        try:
            entries_data[entry_name]['Family'] = entry.annotations['taxonomy'][6]
        except:
            entries_data[entry_name]['Family'] = ""
        
        entries_data[entry_name]['Length'] = len(entry)
    
    
        entries_data[entry_name]['Reference_title'] = ""
        entries_data[entry_name]['Submitters'] = ""
        entries_data[entry_name]['PUBMED_ID'] = ""
        
        if len(entry.annotations['references']) == 1:
            entries_data[entry_name]['Reference_title'] = entry.annotations['references'][0].title
            entries_data[entry_name]['Submitters'] = entry.annotations['references'][0].authors
            entries_data[entry_name]['PUBMED_ID'] = entry.annotations['references'][0].pubmed_id
        else:
            for reference in entry.annotations['references'][:-1]:
                entries_data[entry_name]['Reference_title'] += (reference.title + ';')
                entries_data[entry_name]['Submitters'] += (reference.authors  + ';')
                entries_data[entry_name]['PUBMED_ID'] += (reference.pubmed_id  + ';')
    
            entries_data[entry_name]['Reference_title'] = entries_data[entry_name]['Reference_title'].strip(";")
            entries_data[entry_name]['Submitters'] = entries_data[entry_name]['Submitters'].strip(";")
            entries_data[entry_name]['PUBMED_ID'] = entries_data[entry_name]['PUBMED_ID'].strip(";")
        #for reference in entry.annotations['references']:
        #    if reference.title == 'Direct Submission':
        #        entries_data[entry_name]['Submitters'] = reference.authors
    
        count_cds = 0
        for feature in entry.features:
            if feature.type == 'source':
                for qualif, colname in zip(source_qualifiers, source_qualifiers_columns):
                    if qualif in feature.qualifiers:
                            if qualif == "collection_date":
                                entries_data[entry_name][colname] = standardize_date(feature.qualifiers[qualif][0])
                            else:
                                entries_data[entry_name][colname] = feature.qualifiers[qualif][0]
                    else:
                        entries_data[entry_name][colname] = 'NA'
                if 'environmental_sample' in feature.qualifiers:
                    entries_data[entry_name]['Environmental'] = 'Yes'
                else:
                    entries_data[entry_name]['Environmental'] = 'No'
                    
            if feature.type == 'CDS':
                
                count_cds +=1
                
        entries_data[entry_name]['CDS count'] = count_cds
    
        artificial_condition0 = ('patent' in entry.annotations['references'][-1].journal.lower())
        artificial_condition1 = ("nonfunctional" in entry.description.lower())
    
        if artificial_condition0 or artificial_condition1:
    
            entries_data[entry_name]['Artificial'] = "Yes"
        else:
            entries_data[entry_name]['Artificial'] = "No"
    
    
        if 'UNVERIFIED' in entry.description:
            entries_data[entry_name]['UNVERIFIED'] = "Yes"
        else:
            entries_data[entry_name]['UNVERIFIED'] = "No"
meta_bastrodataframe = pd.DataFrame.from_dict(entries_data, orient='index')

C:\Users\alxva\AppData\Local\Temp\ipykernel_21476\146039531.py:9: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  meta = pd.read_csv(meta_path,sep="\t")


In [22]:
meta_bastrodataframe

,LOCUS,Version,GenBank title,Release date,Organism name,Virus Lineage,Species,Family,Length,Reference_title,...,Isolate,Geo location,Country,Collection date,Tissue/Specimen/Source,Host,Environmental,CDS count,Artificial,UNVERIFIED
PQ541176,PQ541176,PQ541176.1,MAG: Rodent Bastrovirus isolate Mu. musculus_a...,20-FEB-2025,rodent Bastrovirus,Viruses,NA,,5750,Direct Submission,...,Mu. musculus_astro,Sierra Leone,NA,2023,NA,Mus musculus,Yes,2,No,No
OR710300,OR710300,OR710300.1,Rodent Bastrovirus strain CHN/YN/LH-6/2022 non...,08-NOV-2023,rodent Bastrovirus,Viruses,NA,,5893,Direct Submission,...,DhLh72022,China,NA,2022-07,NA,Rattus tanezumi,No,2,No,No


In [25]:
meta_all = pd.concat([meta,meta_bastrodataframe])

In [28]:
meta_all["Accession"] = meta_all.index

In [29]:
meta_all

,LOCUS,Version,GenBank title,Release date,Organism name,Virus Lineage,Species,Family,Length,Reference_title,...,Host_species,Host_genus,Host_family,Host_order,Host_class,Host_phylum,Host_kingdom,Species_ICTV,virus_name,Accession
PZ195463,PZ195463,PZ195463.1,"Goose astrovirus isolate G29, complete genome",23-JUN-2026,Goose astrovirus,Viruses;Riboviria;Orthornavirae;Pisuviricota;S...,NaN,Astroviridae,7116,Direct Submission,...,Gallus gallus,Gallus,Phasianidae,Galliformes,Aves,Chordata,Metazoa,NaN,NaN,PZ195463
PZ195462,PZ195462,PZ195462.1,"Goose astrovirus isolate X-1, complete genome",23-JUN-2026,Goose astrovirus,Viruses;Riboviria;Orthornavirae;Pisuviricota;S...,NaN,Astroviridae,7116,Direct Submission,...,Anser anser,Anser,Anatidae,Anseriformes,Aves,Chordata,Metazoa,NaN,NaN,PZ195462
PZ195461,PZ195461,PZ195461.1,"Goose astrovirus isolate G292, complete genome",23-JUN-2026,Goose astrovirus,Viruses;Riboviria;Orthornavirae;Pisuviricota;S...,NaN,Astroviridae,7116,Direct Submission,...,Anser anser,Anser,Anatidae,Anseriformes,Aves,Chordata,Metazoa,NaN,NaN,PZ195461
PZ195460,PZ195460,PZ195460.1,"Goose astrovirus isolate G259, complete genome",23-JUN-2026,Goose astrovirus,Viruses;Riboviria;Orthornavirae;Pisuviricota;S...,NaN,Astroviridae,7116,Direct Submission,...,Anser anser,Anser,Anatidae,Anseriformes,Aves,Chordata,Metazoa,NaN,NaN,PZ195460
PZ195459,PZ195459,PZ195459.1,"Goose astrovirus isolate G253, complete genome",23-JUN-2026,Goose astrovirus,Viruses;Riboviria;Orthornavirae;Pisuviricota;S...,NaN,Astroviridae,7116,Direct Submission,...,Anser anser,Anser,Anatidae,Anseriformes,Aves,Chordata,Metazoa,NaN,NaN,PZ195459
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Z66541,Z66541,Z66541.1,Human astrovirus type 8 orf2 gene for capsid p...,30-OCT-1995,Human astrovirus 8,Viruses;Riboviria;Orthornavirae;Pisuviricota;S...,Mamastrovirus hominis,Astroviridae,2434,The sequence of human astrovirus serotype 8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Z66541
Z46658,Z46658,Z46658.1,Human astrovirus type 6 gene for capsid polypr...,14-NOV-1994,Human astrovirus 6,Viruses;Riboviria;Orthornavirae;Pisuviricota;S...,Mamastrovirus hominis,Astroviridae,2418,Prevalence of human astrovirus serotype 4: cap...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Z46658
L13745,HUANSSPS,L13745.1,"Human astrovirus serotype 2, complete sequence...",08-MAR-1994,Human astrovirus,Viruses;Riboviria;Orthornavirae;Pisuviricota;S...,NaN,Astroviridae,6828,RNA sequence of astrovirus: distinctive genomi...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,L13745
PQ541176,PQ541176,PQ541176.1,MAG: Rodent Bastrovirus isolate Mu. musculus_a...,20-FEB-2025,rodent Bastrovirus,Viruses,NA,,5750,Direct Submission,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PQ541176


In [26]:
meta_all.to_csv("meta.tsv",sep="\t")

Get coding sequences for future hmmsearch

In [24]:
#======= Analyses CDS feature which annotation does not correspond to target CDS =======
def analyze_cds(record, feature, transl_table=1):
    """
    Extracts translation and coordinates from a CDS feature.
    Returns a dict with details, including internal stop count.
    """
    
    codon_start = int(feature.qualifiers.get("codon_start", [1])[0])
    offset = codon_start - 1


    # Get parts – if not a compound location, treat as single part
    if len(feature.location.parts) > 1:
        parts = feature.location.parts
    else:
        parts = [feature.location]

    
    candidates = []
    for idx, part in enumerate(parts):
        raw_seq = part.extract(record.seq)
        # codon_start offset only applies to the first part
        if idx == 0:
            cds_seq = raw_seq[offset:]
        else:
            cds_seq = raw_seq

        # Translate, include stop codon
        translation = cds_seq.translate(table=transl_table, to_stop=False)
        annotated_translation = feature.qualifiers.get("translation", [""])[0]
        
        # Count internal stops (* not at last position)
        stops = [i for i, aa in enumerate(translation) if aa == "*" and i != len(translation)-1]
        internal_stop_count = len(stops)
        
        if translation and translation[-1] == "*":
            translation = translation[:-1]

        if len(parts) == 1 and translation != annotated_translation:
            print(f"For {record.id} the annotated translation does not correspond to actual translation of coding sequence")
            print("Actual translation: " + translation)
            print("Annotated translation: " + annotated_translation)


        start = int(part.start)
        end = int(part.end)
        
        cand = {
            "record_name": record.name,
            "accession": record.id,
            "location": str(part),                # individual part location
            "codon_start": str(codon_start if idx == 0 else 1),
            "start": start,
            "end": end,
            "strand": part.strand,
            "gene": feature.qualifiers.get("gene", ["unknown"])[0],
            "product": feature.qualifiers.get("product", ["unknown"])[0],
            "transl_table": transl_table,
            "raw_nucleotide": str(raw_seq),
            "cds_sequence": str(cds_seq),
            "translation": str(translation),
            "translation_annot": str(annotated_translation),
            "internal_stop_count": internal_stop_count,
            "is_clean": internal_stop_count == 0,
            "part_index": idx,   # optional, for debugging
        }
        candidates.append(cand)
    return candidates

#======= Validation and correction of frame in CDS feature which annotation does not correspond to target CDS =======
def correct_cds_frame(cds_info, max_stops=2):
    """
    Attempt to correct the translation frame of a CDS.
    Tries all 3 reading frames on the raw nucleotide sequence.
    Returns the best translation and the offset to apply.

    Parameters:
    cds_info - dictionary with information of CDS feature, output of analyze_cds funciton
    max_stops - maximal number of stop codons
    """
    raw = cds_info["raw_nucleotide"]  # use the untrimmed raw sequence
    transl_table = cds_info["transl_table"]
    
    best = None
    best_offset = None
    
    for frame in range(3):  # 0, 1, 2
        seq = Seq(raw[frame:])
        
        trans = seq.translate(table=transl_table, to_stop=False)
        # count internal stops
        internal_stops = [i for i, aa in enumerate(trans) if aa == "*" and i != len(trans)-1]
        stop_count = len(internal_stops)
        
        # We prefer fewer stops, and if equal, longer length
        if stop_count <= max_stops:
            if best is None or (stop_count < best["stop_count"]) or \
               (stop_count == best["stop_count"] and len(trans) > len(best["translation"])):
                best = {
                    "translation": str(trans),
                    "stop_count": stop_count,
                    "frame_offset": frame,  # number of bases to skip from the raw start
                    "nucleotide_used": str(seq),
                }
                best_offset = frame
    return best, best_offset
# ======= write CDS with no target or conflicting annotation
def write_fasta_and_mapping(candidates, fasta_file, mapping_file):
    """
    Writes two files:
      - fasta_file: amino acid sequences with header containing accession and some metadata.
      - mapping_file: TSV (or CSV) with detailed coordinates/strand per candidate.
    """
    with open(fasta_file, "w") as f_fasta, open(mapping_file, "w", newline="") as f_map:
        writer = csv.writer(f_map, delimiter="\t")
        writer.writerow(["candidate_id",
            "record_name", "accession", "start", "end", "codon_start","strand",
            "gene", "product", "is_corrected", "correction_offset", "original_translation"
        ])
        for idx, c in enumerate(candidates, 1):
            # FASTA header: include ID and some quick info
            cid = f"cand_{idx:04d}"
            header = f">{cid}|{c['accession']}|{c['gene']}|start={c['start']}|end={c['end']}|codon_start={c['codon_start']}|strand={c['strand']}"
            f_fasta.write(header + "\n")
            trans = c['translation']
            for i in range(0, len(trans), 60):
                f_fasta.write(trans[i:i+60] + "\n")
            
            # Write mapping line
            writer.writerow([
                cid,
                c['record_name'],
                c['accession'],
                c['start'],
                c['end'],
                c['codon_start'],
                c['strand'],
                c['gene'],
                c['product'],
                c.get('is_corrected', False),
                c.get('correction_frame_offset', ''),
                c.get('original_translation', '')
            ])

In [29]:
cds_candidates = []
cds_counts = []
for rec in recs_noamb:
    cds_count = 0
    
    rec_name = rec.id.split(".")[0]

    for feature in rec.features:
        if feature.type == 'CDS':
            cds_count +=1
            #print(feature)
            cds_infos = analyze_cds(rec, feature)
            for cds_info in cds_infos:
                if not cds_info["is_clean"]:
                    corrected, offset = correct_cds_frame(cds_info, max_stops=1)
                    if corrected is not None:
                        cds_info["is_corrected"] = True
                        cds_info["corrected_translation"] = corrected["translation"]
                        cds_info["original_translation"] = cds_info["translation"]
                        cds_info["translation"] = corrected["translation"]
                        cds_info["correction_frame_offset"] = offset
                        cds_info["codon_start"] = int(cds_info["codon_start"]) + offset
                    else:
                        cds_info["is_corrected"] = False
                        cds_info["correction_failed"] = True
                cds_candidates.append(cds_info)
    print(f"{rec.name} has {cds_count} annotated CDS")
    cds_counts.append(cds_count)
    print("_______")

out_folder = "cds"
out_file_name_temp = "bastro"
#os.mkdir(out_folder)
# Write all CDS

fasta_cand = os.path.join(out_folder, out_file_name_temp + '_cds.faa')
mapping_cand = os.path.join(out_folder, out_file_name_temp + '_cds.tsv')
write_fasta_and_mapping(cds_candidates, fasta_cand, mapping_cand)

For PV605168.1 the annotated translation does not correspond to actual translation of coding sequence
Actual translation: LDAVNGRGHTPKQLLWLQNPDVNHFQAVAVEGLLPLLPCDVIGSVDTNAFVGTLPAVLHAVNAVPGQVGALINHGPLKPQLIACVVHETAHVHCPPLDVAEVTTNVNRALTMPVSMVVSEGVQCAHHGMRVCDAILEHEVAPQNLEEVFLLLAVEHERNVQGIADSACHCTAEACAVLTFPVAALGVREGGGAGERSM
Annotated translation: MDAVNGRGHTPKQLLWLQNPDVNHFQAVAVEGLLPLLPCDVIGSVDTNAFVGTLPAVLHAVNAVPGQVGALINHGPLKPQLIACVVHETAHVHCPPLDVAEVTTNVNRALTMPVSMVVSEGVQCAHHGMRVCDAILEHEVAPQNLEEVFLLLAVEHERNVQGIADSACHCTAEACAVLTFPVAALGVREGGGAGERSM
For PV605168.1 the annotated translation does not correspond to actual translation of coding sequence
Actual translation: LHIPFVLNREEKEYLLKVLRCDLVFEDRVAHTHSMVRALHAFANHHAYRHCEGAIDIGGDFRNIKRRAVHVCGLVDNARDELRLQRAMVDQRPNLAGYCIHGVENCRERADKGICVHAAYDITWQQWQQAFDSHGLKVVDVWILEPEELFGRVTPPIYGIQMHRLANRWAMLLDDACAGYVHDDFEWRKYRETGGTIGPRFNIGCSPEGGWGPLRRYTLFRCVHPFRQITLRHPVDRNTVHVPTPAGHVISCPRSVWQNLLSWACSRADNKFDFSSLLAYARALRQRLTIGGQTVQEGWSQDPRQVHELCIACFVMAAAMRYTRTQNIAVAFAEL

Scan obtained CDS against PFAM

In [62]:
import copy
def load_domain_map(csv_file):
    """
    Load domain-to-ORF mapping from a CSV file.
    Expected columns: domain_name, orf_type
    Returns a dict {domain_name: orf_type}.
    """
    domain_map = {}
    with open(csv_file, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            domain = row['domain_name'].strip()
            orf = row['orf_type'].strip()
            if domain and orf:
                domain_map[domain] = orf
    return domain_map

def parse_hmmscan_domtbl(domtbl_file, evalue_threshold=0.01):
    """
    Parse hmmscan --domtblout output.
    Returns a dict: {query_id: [list_of_domain_hits]}
    Each hit is a dict with keys:
        'domain_name', 'domain_accession', 'domain_description',
        'query_start', 'query_end', 'score', 'evalue', 'bitscore'
    """
    domain_hits = {}
    with open(domtbl_file) as f:
        for line in f:
            if line.startswith('#'):
                continue
            fields = line.strip().split()
            if len(fields) < 23:
                continue
            # Fields (0‑based indices):
            # 0: target name (domain)
            # 1: target accession
            # 2: tlen
            # 3: query name (your sequence ID)
            # 4: query accession (often '-')
            # 5: qlen
            # 6: evalue (full sequence)
            # 7: score (full sequence)
            # 8: bias
            # 9: domain number
            # 10: domain count in this hit
            # 11: domain evalue
            # 12: domain score
            # 13: domain bias
            # 14: hmm coord from
            # 15: hmm coord to
            # 16: ali coord from (query start)
            # 17: ali coord to (query end)
            # 18: env coord from
            # 19: env coord to
            # 20: acc (optional)
            # 21: description of target (the rest)
            query = fields[3]
            domain_name = fields[0]
            domain_acc = fields[1]
            domain_evalue = float(fields[11])
            if domain_evalue > evalue_threshold:
                continue
            query_start = int(fields[16])
            query_end = int(fields[17])
            bitscore = float(fields[13])
            # Full sequence description (may contain spaces)
            desc = ' '.join(fields[22:]) if len(fields) > 22 else ''

            
            hit = {
                'domain_name': domain_name,
                'domain_accession': domain_acc,
                'domain_description': desc,
                'query_start': query_start,
                'query_end': query_end,
                'score': bitscore,
                'evalue': domain_evalue
            }
            domain_hits.setdefault(query, []).append(hit)
    return domain_hits

def assign_orf_from_domains(domain_hits, domain_map, min_score=0, min_coverage=0.5):
    """
    Given a list of domain hits for a single sequence,
    return the most likely ORF type based on the best hit (lowest E-value).
    If no hit matches the map, return None.
    """
    if not domain_hits:
        return None
    # Sort by E-value (ascending)
    sorted_hits = sorted(domain_hits, key=lambda x: x['evalue'])
    domains = []
    coords = {}
    for hit in sorted_hits:
        dom_name = hit['domain_name']
        if dom_name in domain_map:
            domains.append(domain_map[dom_name])
            if domain_map[dom_name] == "1B" or domain_map[dom_name] == "MetTransf":
                #print(hit)
                coords[domain_map[dom_name]] = [hit["query_start"],hit["query_end"]]
            
                
    domains = list(set(domains))
    if len(domains) > 1:
        print("Two domains are found in one CDS")
        print(domains)
        return coords
    if len(domains) == 1:
        if domains[0] == '1B':
            coords['1B'] = [hit["query_start"],hit["query_end"]]
            return coords
        else:
            return domains[0]
    else:
        return None


def assign_problematic_orfs(candidate_tsv, domain_table_file, domain_map_file, output_tsv):
    """
    For each problematic candidate, use the domain hits to assign an ORF type.
    Writes a new TSV with an additional 'assigned_orf' column.
    """
    domain_map = load_domain_map(domain_map_file)
    domain_hits = parse_hmmscan_domtbl(domain_table_file)
    with open(candidate_tsv) as f:
        reader = csv.DictReader(f, delimiter='\t')
        rows = list(reader)

    not_assigned_rows = []
    assigned_rows = []
    for row in rows:
        # Reconstruct the FASTA header from the candidate info
        cid = row['candidate_id']
        header = f"{cid}|{row['accession']}|{row['gene']}|start={row['start']}|end={row['end']}|codon_start={row['codon_start']}|strand={row['strand']}"
        hits = domain_hits.get(header, [])
        orf_type = assign_orf_from_domains(hits, domain_map)
        if type(orf_type) is dict:
            for orf,coords in orf_type.items():
                row1 = copy.deepcopy(row)
                row1['assigned_orf'] = orf
                row1['dom_start'] = coords[0]
                row1['dom_end'] = coords[1]
                assigned_rows.append(row1)
        else:    
            row['assigned_orf'] = orf_type if orf_type else 'unknown'
            assigned_rows.append(row)
        if not orf_type:
            not_assigned_rows.append(row)
        
    print(f"{len(not_assigned_rows)} CDS were not assigned")
    
    # Write updated TSV
    with open(output_tsv, 'w', newline='') as f:
        fieldnames = reader.fieldnames + ['assigned_orf', 'dom_start', 'dom_end']
        writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter='\t')
        writer.writeheader()
        writer.writerows(assigned_rows)

In [63]:
assign_problematic_orfs("cds/bastro_cds.tsv", "hmmer/bastro_cds_domains.tbl", "hmmer/domain_map.csv", "hmmer/bastro_cds_assigned.tsv")

Two domains are found in one CDS
['MetTransf', '1B']
Two domains are found in one CDS
['MetTransf', '1B']
Two domains are found in one CDS
['MetTransf', '1B']
Two domains are found in one CDS
['MetTransf', '1B']
Two domains are found in one CDS
['MetTransf', '1B']
Two domains are found in one CDS
['MetTransf', '1B']
Two domains are found in one CDS
['MetTransf', '1B']
Two domains are found in one CDS
['MetTransf', '1B']
Two domains are found in one CDS
['MetTransf', '1B']
Two domains are found in one CDS
['MetTransf', '1B']
Two domains are found in one CDS
['MetTransf', '1B']
Two domains are found in one CDS
['MetTransf', '1B']
Two domains are found in one CDS
['MetTransf', '1B']
Two domains are found in one CDS
['MetTransf', '1B']
Two domains are found in one CDS
['MetTransf', '1B']
10 CDS were not assigned


## Retrieve protein sequences of assigned ORFs

In [116]:
records_cds = SeqIO.parse("cds/bastro_cds.faa", "fasta")
assigned_tsv = pd.read_csv("hmmer/bastro_cds_assigned.tsv", sep="\t") 
meta = pd.read_csv("meta.tsv",sep="\t")

meta = meta.set_index("Accession")
#assigned_tsv = assigned_tsv.set_index("record_name")


recs_RdRp = []
recs_capsid = []

for rec in records_cds:
    recid = rec.id.split("|")[1].split(".")[0]
    cid = rec.id.split("|")[0]
    organism = meta.loc[recid]["Organism name"].replace(" ", "-")
    host = meta.loc[recid]["host_clean"]
    if type(host) == str:
        host = host.replace(" ", "-")
    else:
        host = "NA"
    rec.name = recid + "|" + organism + "|" +  host + "|" + cid
    rec.id = rec.name
    rec.description=""
    
    
    cand_tsv = assigned_tsv[assigned_tsv["candidate_id"] == cid]
    print("______________")
    print(cand_tsv)
    if len(cand_tsv) == 1:
        domain = cand_tsv["assigned_orf"].iloc[0]
        if domain == "1B":
            st = int(cand_tsv["dom_start"].iloc[0])
            e = int(cand_tsv["dom_end"].iloc[0])
            if e < st:
                st1 = st
                st = e
                e = st1
            rec.seq = rec.seq[st-1:e]
            
            recs_RdRp.append(rec)
            print(f"For {rec.id} appended ORF1b with coordinates {st}-{e}")
        elif domain == "2" or domain == "CaliciCapsid":
            recs_capsid.append(rec)
            print(f"For {rec.id} appended capsid")
    else:
        print("Several proteins")
        for row in cand_tsv.iterrows():
            domain = row[1]["assigned_orf"]
            print(domain)
            if domain == "1B":
                st = int(row[1]["dom_start"])
                e = int(row[1]["dom_end"])
                if e < st:
                    st1 = st
                    st = e
                    e = st1
                rec.seq = rec.seq[st-1:e]
                recs_RdRp.append(rec)
                print(f"For {rec.id} appended ORF1b with coordinates {st}-{e}")
            elif domain == "2" or domain == "CaliciCapsid":
                recs_capsid.append(rec)
                print(f"For {rec.id} appended capsid")

SeqIO.write(recs_RdRp, "cds_assigned/bastro_rdrp.fasta", "fasta")
SeqIO.write(recs_capsid, "cds_assigned/bastro_capsid.fasta", "fasta")

______________
  candidate_id record_name   accession  start  end  codon_start  strand  gene  \
0    cand_0001    PV605168  PV605168.1      0  595            1      -1  ORF3   

                product  is_corrected  correction_offset  \
0  hypothetical protein         False                NaN   

   original_translation assigned_orf  dom_start  dom_end  
0                   NaN      unknown        NaN      NaN  
______________
  candidate_id record_name   accession  start   end  codon_start  strand  \
1    cand_0002    PV605168  PV605168.1    112  3688            1       1   
2    cand_0002    PV605168  PV605168.1    112  3688            1       1   

   gene                              product  is_corrected  correction_offset  \
1  ORF1  putative non-structural polyprotein         False                NaN   
2  ORF1  putative non-structural polyprotein         False                NaN   

   original_translation assigned_orf  dom_start  dom_end  
1                   NaN           1B

28

In [74]:
cand_tsv["assigned_orf"].iloc[0]

'2'

In [65]:
cand_tsv.index

Int64Index([3], dtype='int64')

In [51]:
row[1]

candidate_id                                      cand_0002
record_name                                        PV605168
accession                                        PV605168.1
start                                                   112
end                                                    3688
codon_start                                               1
strand                                                    1
gene                                                   ORF1
product                 putative non-structural polyprotein
is_corrected                                          False
correction_offset                                       NaN
original_translation                                    NaN
assigned_orf                                             1B
dom_start                                             432.0
dom_end                                               804.0
Name: 1, dtype: object

In [16]:
cand_tsv.iterrows()

<generator object DataFrame.iterrows at 0x000002A40115AC00>

In [9]:
rec.seq

Seq('LHIPFVLNREEKEYLLKVLRCDLVFEDRVAHTHSMVRALHAFANHHAYRHCEGA...PVE')